# 03_04 Many heads

*📖 Book: section 3.6*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** One head gives one way of looking. Several heads can each look for something different; our `model.py` has just one.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why one way of looking may not be enough
- How to build a tiny 2-head attention and join the heads
- That splitting into heads doesn't add weights
- That our `model.py` has ONE head, and the full GPT of chapter 4 has several

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **head** | a word with three uses: an **attention head** (one complete attention step with its own query, key and value layers), `self.head` (the last layer of `model.py`, which makes the 137 scores), and the **output head** (the same last layer, in a general GPT) | our `model.py` has one of each |
| **multi-head attention** | several heads running side by side on the same letters, their results joined together | 2 heads of 16 numbers |
| **concatenate** | join rows end to end into a longer row (`torch.cat`) | two rows of 16 numbers make one row of 32 |

## 3. The idea

### ❓ Why would one head not be enough?

A head makes **one** set of attention weights per letter, so one way of looking. But guessing a letter may need several things at once: the letter just before, the start of this word, the last vowel. With one head, the model has to squeeze all of that into one mix. With several heads, each can search for its own thing. Take `μοῦσα`:

```
head 1: looks at the previous letter      (σ before α)
head 2: looks at the start of the word   (μ)
        └── their results are joined, so the next layer gets both
```

That picture is the *idea*. We haven't measured it in our model. Real models built like this work well, and a fair experiment ([05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb)) could test it for ours.

### ❓ How do several heads fit in the same space?

Each head works in a smaller space. With 32 numbers per letter and 2 heads, each head makes queries, keys and values of 16 numbers. Each head returns a context of 16 numbers per letter, and we **concatenate** the two (join them end to end): 16 + 16 = 32 again. So the result has the same shape as one big head, and it fits wherever one big head fitted.

```
head 1:  (5, 16) ┐
                  ├─ joined →  (5, 32)
head 2:  (5, 16) ┘
```

👉 **Predict first:** will 2 heads of 16 have more weights than our 1 head of 32, fewer, or the same?

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: one head, as a function

We pack the attention of 03_02 and 03_03 (scores, mask, softmax, mix) into one function, so we can call it once per head. One difference: each head divides by `√16`, the size of its **own** queries, not `√32`. We load our model only to borrow the real embedding rows of `ἄνδρα`.

In [ ]:
import torch

from homer_ai.lessons import load_homer_model

# read only: we load the saved model, we never save
tokenizer, model = load_homer_model(fallback_untrained=True)

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.lessons import draw_weights  # the heatmap function, drawn in 03_01
from homer_ai.studio_style import studio_figure

In [ ]:
from torch import nn


def one_head(x, query, key, value):
    Q, K, V = query(x), key(x), value(x)
    scores = (Q @ K.T) / (Q.shape[-1] ** 0.5)  # Q.shape[-1] is the head's size
    mask = torch.triu(torch.ones(len(x), len(x)) * float("-inf"), diagonal=1)
    weights = torch.softmax(scores + mask, dim=-1)
    return weights, weights @ V


ids = torch.tensor(tokenizer.encode("ἄνδρα"))
with torch.no_grad():
    x = model.embedding(ids)
print("x", tuple(x.shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`x` is `(5, 32)`, the real embedding rows of `ἄνδρα`.

</details>

### ❓ Step 2: build two heads and join them

Each head has its own three layers, going from 32 numbers down to 16. We make two heads, run each one, and join the results. These heads are **untrained** (random numbers, seeded so you get the same ones), so all they show is how the pieces fit.

👉 **Predict first:** what shapes will one head's weights and context have, and the joined result?

In [ ]:
torch.manual_seed(0)
head_size = 16
heads = [
    (
        nn.Linear(32, head_size, bias=False),  # query
        nn.Linear(32, head_size, bias=False),  # key
        nn.Linear(32, head_size, bias=False),  # value
    )
    for _ in range(2)
]

all_weights, all_contexts = [], []
with torch.no_grad():
    for query, key, value in heads:
        w, c = one_head(x, query, key, value)
        all_weights.append(w)
        all_contexts.append(c)

print("one head: weights", tuple(all_weights[0].shape))
print("one head: context", tuple(all_contexts[0].shape))
joined = torch.cat(all_contexts, dim=-1)  # join end to end, along the numbers
print("joined:", tuple(joined.shape))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Each head gives `(5, 5)` weights and a `(5, 16)` context. Joined, they make `(5, 32)`, the same shape as our single head's context.

</details>

### ❓ Do the two heads really look in different ways?

Here are the two sets of weights side by side. Remember these heads are random and untrained, so any difference is just chance. In training, each head would be pushed towards something useful, and the heads would be pushed to differ.

👉 **Predict first:** will the two heatmaps be identical?

In [ ]:
fig = studio_figure("Two heads", "Each head makes its own attention pattern", (12, 6.2))
letters = list("ἄνδρα")
left, right = all_weights[0].numpy(), all_weights[1].numpy()
draw_weights(fig, (0.09, 0.08, 0.38, 0.58), left, letters, "head 1 (untrained)")
draw_weights(fig, (0.56, 0.08, 0.38, 0.58), right, letters, "head 2 (untrained)")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Both are triangles (each head has the mask), and the numbers differ a little (the 2nd row, for example, is `0.49 / 0.51` against `0.51 / 0.49`). Untrained heads spread their shares almost evenly, so the difference is small. After training, each head would be pushed to look in its own, sharper way.

</details>

### ❓ How many weights do many heads cost?

You might expect twice the heads to cost twice the weights. Let's count them with real layers: 1, 2, 4 and 8 heads, always splitting the 32 numbers among the heads, and compare with our model's own one head.

👉 **Predict first:** the answer for 2 heads. Is it more than our model's 3,072?

In [ ]:
for n_heads in [1, 2, 4, 8]:
    size = 32 // n_heads
    layers = [nn.Linear(32, size, bias=False) for _ in range(3 * n_heads)]
    total = sum(layer.weight.numel() for layer in layers)
    print(f"{n_heads} head(s) of {size:>2} numbers: {total:,} weights")

ours = sum(layer.weight.numel() for layer in (model.query, model.key, model.value))
print(f"our model.py (1 head of 32): {ours:,} weights")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

3,072 every time, the same as our model's one head. Several smaller heads cost the **same** weights as one big one, because each is smaller by exactly as much as there are more of them.

</details>

### ❓ Your turn: what would 16 heads cost?

The table stops at 8 heads. Work out the number for 16 heads yourself, with plain arithmetic.

👉 **Predict first:** more or fewer weights than 8 heads?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
n_heads = 16
head_size = 32 // n_heads

# 👉 do: the weights of 16 heads: each head has a query, a key and a value layer,
#        and each layer takes 32 numbers in and gives head_size numbers out (no bias)
weights_16_heads = None

In [ ]:
from homer_ai.lessons import check

check(
    "weights of 16 heads",
    weights_16_heads,
    16 * 3 * 32 * (32 // 16),
    coach="One layer from 32 numbers to head_size numbers has 32 × head_size "
    "weights. A head has 3 of them, and there are 16 heads.",
)

<details><summary>💡 Hint</summary>

Multiply four numbers: the heads, the 3 layers per head, the 32 numbers going in, and `head_size` coming out.

</details>

<details><summary>✅ Answer</summary>

```python
n_heads = 16
head_size = 32 // n_heads

weights_16_heads = n_heads * 3 * 32 * head_size
```

3,072 again. Twice the heads, each half the size, so nothing changes. What does change is how little room each head has: with 16 heads, a head works with only 2 numbers per letter.

</details>

### ❓ Does the joined result still work with our model?

Yes. It's `(5, 32)`, which is what `model.head` expects. We feed it in just to see the shape (the numbers mean nothing, because these heads are untrained).

👉 **Predict first:** what shape will the output have?

In [ ]:
with torch.no_grad():
    out = model.head(joined)
print(tuple(out.shape), "= 5 letters × 137 scores")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(5, 137)`. So joined heads can replace our single head without changing anything after it.

</details>

### ❓ Does our `model.py` have many heads?

No, just **one**. Its three layers are 32 × 32 and it makes one set of weights. The GPT of [chapter 4](../04_building_the_model/04_00_chapter_overview.ipynb) (`src/homer_ai/gpt.py`) has several. `model.py` stays the small one-head model.

## 5. Recap

- Several heads each look for something different; each works in a smaller space and their results are joined (`torch.cat`) back to the full size.
- Splitting 32 numbers into heads costs the same weights: 3,072 for 1, 2, 4 or 8 heads.
- Our `model.py` has one head; the GPT of chapter 4 has several (4 in the shipped one).

**You can now:**

- run several heads over the same letters and join their results with `torch.cat`
- work out what any number of heads costs in weights

**Next:** [04_00: Building the model](../04_building_the_model/04_00_chapter_overview.ipynb): the full model around attention.